### 1. Overview
Advanced Search Techniques with Azure AI Search: Keyword, Vector, and Hybrid Methods

This notebook demonstrates how to perform different types of searches using Azure AI Search, including keyword search, vector search, hybrid search, semantic ranking, and query rewriting.

### 2. Set Up Environment Variables
Just like for Journey 1, create the `.env` file in the same directory as this notebook and update the variables.
You can use the `.env.sample` file to see which variables are needed.

After setting up, the notebook will automatically load these values using dotenv.

### 3. Load Environment Variables

Run the following command to load environment variables from the .env file:

In [5]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(override=True) # take environment variables from .env.

endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
index_name = os.environ["AZURE_SEARCH_INDEX_NAME"]
credential = AzureKeyCredential(os.getenv("AZURE_SEARCH_ADMIN_KEY")) if os.getenv("AZURE_SEARCH_ADMIN_KEY") else DefaultAzureCredential()

This will ensure all necessary credentials are available before setting up the API client.

### 4. Set Up API Client and Define the Display Function

Initialize the Azure AI Search Client for interacting with the Azure Search service and make the search results easier to read by defining a function that formats and displays results:

In [6]:
from azure.search.documents import SearchClient
import pandas as pd

search_client = SearchClient(endpoint, index_name, credential)

def display_results(results):
    df = pd.json_normalize(list(results)).dropna(axis=1, how='all')
    df["chunk"] = df["chunk"].apply(lambda c: c[:300] + '...' if len(c) > 300 else c)
    first_cols = ['title', 'chunk', '@search.score']
    df = df[first_cols + [col for col in df.columns if col not in first_cols]]

    df = df.style.set_properties(**{
        'max-width': '500px',
        'text-align': 'left',
        'white-space': 'normal',
        'word-wrap': 'break-word'
    }).hide(axis="index")


    return df


### 5. Perform Different Search Methods

#### Keyword Search

Execute a traditional keyword-based search:

In [7]:
results = search_client.search(search_text="What is Contoso", top=5, select=["title", "chunk"])

display_results(results)


title,chunk,@search.score
Northwind_Health_Plus_Benefits_Details.pdf,"the tips outlined above, you can help ensure that your request for services or treatments is approved in a timely manner and that you are receiving the most appropriate care. The Group And You OTHER INFORMATION ABOUT THIS PLAN The Group and You The Northwind Health Plus plan is a gro...",4.684504
Northwind_Standard_Benefits_Details.pdf,"At Contoso, we understand that medical costs can be intimidating and confusing, which is why we’ve partnered with Northwind Health to offer our employees the Northwind Standard plan. This plan provides a balance billing protection, meaning that you are protected from unexpected costs when visi...",4.370165
Northwind_Standard_Benefits_Details.pdf,"providers that are not available from participating providers. Additionally, in some cases, the health plan may cover non-participating providers’ charges if there are no participating providers in your area. Tips In order to avoid costly balance billing amounts, it is important to make sure...",3.893067
Northwind_Health_Plus_Benefits_Details.pdf,"about your health care. With the Northwind Health Plus Plan, you can take advantage of the coverage provided for these services and get the treatment you need. Substance Use Disorder Substance Use Disorder Coverage At Contoso, we are proud to offer our employees Northwind Health Plus, an i...",3.590473
Northwind_Standard_Benefits_Details.pdf,"• Understand any restrictions associated with any government-sponsored programs you may be enrolled in. • Your Northwind Standard plan does not cover certain services, such as emergency care, mental health and substance abuse coverage, or out-of-network services. Be sure to explore alternat...",3.263199


#### Vector Search

Retrieve documents using vector similarity search:

In [8]:
from azure.search.documents.models import VectorizableTextQuery

results = search_client.search(vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")], top=5, select=["title", "chunk"])

display_results(results)

HttpResponseError: () Could not complete vectorization action. Could not reach the vectorization endpoint.
Code: 
Message: Could not complete vectorization action. Could not reach the vectorization endpoint.

#### Hybrid Search (Keyword + Vector Search)

Combine keyword and vector searches for better accuracy:

In [9]:
results = search_client.search(
    search_text="What is Contoso",
    vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")],
    top=5,
    select=["title", "chunk"]
)

display_results(results)

HttpResponseError: () Could not complete vectorization action. Could not reach the vectorization endpoint.
Code: 
Message: Could not complete vectorization action. Could not reach the vectorization endpoint.

#### Hybrid Search + Semantic Ranker

Enhance search results using a semantic ranker:

In [10]:
results = search_client.search(
    search_text="What is Contoso",
    vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")],
    top=5,
    select=["title", "chunk"],
    query_type="semantic",
    semantic_configuration_name="ragtime2-semantic-configuration"
)

display_results(results)

HttpResponseError: (InvalidRequestParameter) Unknown semantic configuration 'ragtime2-semantic-configuration'.
Parameter name: semanticConfiguration
Code: InvalidRequestParameter
Message: Unknown semantic configuration 'ragtime2-semantic-configuration'.
Parameter name: semanticConfiguration
Exception Details:	(UnknownSemanticConfiguration) Unknown semantic configuration 'ragtime2-semantic-configuration'.
	Code: UnknownSemanticConfiguration
	Message: Unknown semantic configuration 'ragtime2-semantic-configuration'.

#### Hybrid Search + Semantic Ranker + Query Rewriting

Use semantic ranking and query rewriting for improved relevance:

In [ ]:
results = search_client.search(
    search_text="What is Contoso",
    vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")],
    top=5,
    select=["title", "chunk"],
    query_type="semantic",
    semantic_configuration_name="ragtime2-semantic-configuration",
    query_rewrites="generative",
    query_language="en"
)

display_results(results)

### 6. Challenge
Let's have a look at the data of our search index and try to think how users might ask questions - and with which search query type the relevant chunks would be retrieved best!

1. Review content of the PerksPlus.pdf
2. Formulate two questions that users might ask about this content
3. Make assumptions about which search method will perform better (focus on keyword search vs. vector search)
4. Test the assumption by executing both searches and comparing the retrieved results.



In [ ]:
    question = "Does PerksPlus support reducing anxiety or stress?"
    results_keyword = search_client.search(search_text=question, top=5, select=["title", "chunk"])

    print("Key word search results")
    display_results(results_keyword)

    results_vector = search_client.search(vector_queries=[VectorizableTextQuery(text=question, k_nearest_neighbors=50, fields="text_vector")], top=5, select=["title", "chunk"])

    print("Vector search results")
    display_results(results_vector)

Key word search results


title,chunk,@search.score
PerksPlus.pdf,PerksPlus Health and Wellness Reimbursement Program for Contoso Electronics Employees This document contains information generated using a language model (Azure OpenAI). The information contained in this document is only for demonstration purposes and does not reflect the ...,11.965925
PerksPlus.pdf,"equipment purchases • Sports team fees • Health retreats and spas • Outdoor adventure activities (such as rock climbing, hiking, and kayaking) • Group fitness classes (such as dance, martial arts, and cycling) • Virtual fitness programs (such as online yoga and workout classes) In additi...",9.303395
Northwind_Standard_Benefits_Details.pdf,"the risks and benefits of the treatment with your provider before beginning treatment. Massage Therapy COVERED SERVICES: Massage Therapy At Contoso, we understand the importance of taking time to care for yourself and to reduce stress. That is why Northwind Health offers massage therapy co...",8.995705
Northwind_Health_Plus_Benefits_Details.pdf,"necessary. It also does not cover services provided by non-network providers. Tips for Employees If you or someone you care about is struggling with SUD, there are a few things you can do to get the most out of your Northwind Health Plus plan: • Talk to your doctor or a mental health prof...",7.397019
Northwind_Health_Plus_Benefits_Details.pdf,"accurate and complete information to the review team. • If your coverage is denied, talk to your doctor about appealing the decision. • If you are considering a service or medication that is not covered by Northwind Health Plus, ask your doctor about other options that may be available. Pe...",5.692018


## Troubleshooting

- **Environment Variables Not Loaded:** Ensure you have correctly set the .env file or manually export them in your terminal before running the notebook.
- **Authentication Issues:** If using Managed Identity, make sure your Azure identity has proper role assignments.
- **Search Results Are Empty:** Ensure your Azure AI Search index contains vectorized data.
- **Query Rewriting Issues:** Ensure your search service supports semantic configurations and generative query rewrites.

## Summary

This notebook demonstrates different search techniques using Azure AI Search, including keyword search, vector search, hybrid search, semantic ranking, and query rewriting. The approach enhances search accuracy by leveraging vector embeddings and semantic understanding to retrieve the most relevant documents.

